# Weight preparation notebook for phase 2 training

This notebook runs some basic steps to extract the model weights from a checkpoint (.tar) file into a ready to use .pth file for phase 2 of the experimental design. It is assumed that the checkpoint has been generated by running LDG/main.py for training.

## Basic imports and definitions

In [18]:
from collections import OrderedDict
import torch
import matplotlib.pyplot as plt
import numpy as np
from models.ModifiedLDG import load_base_mLDG
from mmcv.runner import load_state_dict

In [16]:
class RecorderMeter(object):
    """Computes and stores the minimum loss value and its epoch index"""

    def __init__(self, total_epoch):
        self.reset(total_epoch)

    def reset(self, total_epoch):
        self.total_epoch = total_epoch
        self.current_epoch = 0
        self.epoch_losses = np.zeros((self.total_epoch, 2), dtype=np.float32)    # [epoch, train/val]
        self.epoch_accuracy = np.zeros((self.total_epoch, 2), dtype=np.float32)  # [epoch, train/val]

    def update(self, idx, train_loss, train_acc, val_loss, val_acc):
        self.epoch_losses[idx, 0] = train_loss * 30
        self.epoch_losses[idx, 1] = val_loss * 30
        self.epoch_accuracy[idx, 0] = train_acc
        self.epoch_accuracy[idx, 1] = val_acc
        self.current_epoch = idx + 1

    def plot_curve(self, save_path):

        title = 'the accuracy/loss curve of train/val'
        dpi = 80
        width, height = 1800, 800
        legend_fontsize = 10
        figsize = width / float(dpi), height / float(dpi)

        fig = plt.figure(figsize=figsize)
        x_axis = np.array([i for i in range(self.total_epoch)])  # epochs
        y_axis = np.zeros(self.total_epoch)

        plt.xlim(0, self.total_epoch)
        plt.ylim(0, 100)
        interval_y = 5
        interval_x = 5
        plt.xticks(np.arange(0, self.total_epoch + interval_x, interval_x))
        plt.yticks(np.arange(0, 100 + interval_y, interval_y))
        plt.grid()
        plt.title(title, fontsize=20)
        plt.xlabel('the training epoch', fontsize=16)
        plt.ylabel('accuracy', fontsize=16)

        y_axis[:] = self.epoch_accuracy[:, 0]
        plt.plot(x_axis, y_axis, color='g', linestyle='-', label='train-accuracy', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        y_axis[:] = self.epoch_accuracy[:, 1]
        plt.plot(x_axis, y_axis, color='y', linestyle='-', label='valid-accuracy', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        y_axis[:] = self.epoch_losses[:, 0]
        plt.plot(x_axis, y_axis, color='g', linestyle=':', label='train-loss-x30', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        y_axis[:] = self.epoch_losses[:, 1]
        plt.plot(x_axis, y_axis, color='y', linestyle=':', label='valid-loss-x30', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        if save_path is not None:
            fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
            print('Saved figure')
        plt.close(fig)

## AffectNet 7-class checkpoint
Make sure a checkpoint file is present in the CWD or any of its subdirectories. Then make sure the `checkpoint_path` variable is correctly set with a path to the .tar file.

In [ ]:
checkpoint_path = "./weights/7class_best_LDG.tar"

#  Load the checkpoint, extract the state_dict, remove 'module.' prefix if needed, and save as .pth
checkpoint = torch.load(checkpoint_path, map_location="cpu")
state_dict = checkpoint["state_dict"] if "state_dict" in checkpoint else checkpoint

# Remove 'module.' prefix
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k.replace("module.", "")  # remove `module.`
    new_state_dict[name] = v

torch.save(new_state_dict, "./weights/7class_best_LDG.pth")


In [ ]:
# Sanity check to ensure the keys don't have 'module.' prefix
new_checkpoint = torch.load('./weights/7class_best_LDG.pth', map_location='cpu')
print(new_checkpoint.keys())

odict_keys(['input_layer.0.weight', 'input_layer.1.weight', 'input_layer.1.bias', 'input_layer.1.running_mean', 'input_layer.1.running_var', 'input_layer.1.num_batches_tracked', 'input_layer.2.weight', 'body.0.0.res_layer.0.weight', 'body.0.0.res_layer.0.bias', 'body.0.0.res_layer.0.running_mean', 'body.0.0.res_layer.0.running_var', 'body.0.0.res_layer.0.num_batches_tracked', 'body.0.0.res_layer.1.weight', 'body.0.0.res_layer.2.weight', 'body.0.0.res_layer.3.weight', 'body.0.0.res_layer.4.weight', 'body.0.0.res_layer.4.bias', 'body.0.0.res_layer.4.running_mean', 'body.0.0.res_layer.4.running_var', 'body.0.0.res_layer.4.num_batches_tracked', 'body.0.1.res_layer.0.weight', 'body.0.1.res_layer.0.bias', 'body.0.1.res_layer.0.running_mean', 'body.0.1.res_layer.0.running_var', 'body.0.1.res_layer.0.num_batches_tracked', 'body.0.1.res_layer.1.weight', 'body.0.1.res_layer.2.weight', 'body.0.1.res_layer.3.weight', 'body.0.1.res_layer.4.weight', 'body.0.1.res_layer.4.bias', 'body.0.1.res_layer.4

In [ ]:
# Sanity check to load the model with the new weights
# no warnings should be printed by this call
mldg = load_base_mLDG(
        checkpoint_path='./weights/7class_best_LDG.pth', 
        uses_ef_modules=False,
        num_classes=7)

Loading checkpoint from ./weights/7class_best_LDG.pth


## AffectNet 8-class checkpoint
Make sure a checkpoint file is present in the CWD or any of its subdirectories. Then make sure the `checkpoint_path` variable is correctly set with a path to the .tar file.

In [22]:
checkpoint_path = "./weights/8class_best_LDG.tar"

#  Load the checkpoint, extract the state_dict, remove 'module.' prefix if needed, and save as .pth
checkpoint = torch.load(checkpoint_path, map_location="cpu")
state_dict = checkpoint["state_dict"] if "state_dict" in checkpoint else checkpoint

# Remove 'module.' prefix
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k.replace("module.", "")  # remove `module.`
    new_state_dict[name] = v

torch.save(new_state_dict, "./weights/8class_best_LDG.pth")


In [23]:
# Sanity check to ensure the keys don't have 'module.' prefix
new_checkpoint = torch.load('./weights/8class_best_LDG.pth', map_location='cpu')
print(new_checkpoint.keys())

odict_keys(['input_layer.0.weight', 'input_layer.1.weight', 'input_layer.1.bias', 'input_layer.1.running_mean', 'input_layer.1.running_var', 'input_layer.1.num_batches_tracked', 'input_layer.2.weight', 'body.0.0.res_layer.0.weight', 'body.0.0.res_layer.0.bias', 'body.0.0.res_layer.0.running_mean', 'body.0.0.res_layer.0.running_var', 'body.0.0.res_layer.0.num_batches_tracked', 'body.0.0.res_layer.1.weight', 'body.0.0.res_layer.2.weight', 'body.0.0.res_layer.3.weight', 'body.0.0.res_layer.4.weight', 'body.0.0.res_layer.4.bias', 'body.0.0.res_layer.4.running_mean', 'body.0.0.res_layer.4.running_var', 'body.0.0.res_layer.4.num_batches_tracked', 'body.0.1.res_layer.0.weight', 'body.0.1.res_layer.0.bias', 'body.0.1.res_layer.0.running_mean', 'body.0.1.res_layer.0.running_var', 'body.0.1.res_layer.0.num_batches_tracked', 'body.0.1.res_layer.1.weight', 'body.0.1.res_layer.2.weight', 'body.0.1.res_layer.3.weight', 'body.0.1.res_layer.4.weight', 'body.0.1.res_layer.4.bias', 'body.0.1.res_layer.4

In [25]:
# Sanity check to load the model with the new weights
# no warnings should be printed by this call
mldg = load_base_mLDG(
        checkpoint_path='./weights/8class_best_LDG.pth', 
        uses_ef_modules=True,
        num_classes=8)

Loading checkpoint from ./weights/8class_best_LDG.pth
